# Corpus EDA — the archived eight-brick track

This notebook reads the **reports**, not the raw dataset.

`data/reports/01_eda.json` and its siblings are committed; the corpus they
describe is not, because it is not ours to redistribute. So this notebook
runs in a fresh checkout with no downloads, and what it shows is exactly
what the recorded run measured.

Regenerating the reports themselves needs the dataset: `make eda`.

Nothing here trains anything. Training and evaluation logic lives in
`src/`, is imported rather than pasted, and is covered by `tests/`.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REPORTS = ROOT / 'data' / 'reports'

def report(name):
    return json.loads((REPORTS / name).read_text(encoding='utf-8'))

eda = report('01_eda.json')
print({k: v for k, v in eda.items() if isinstance(v, int)})


## 1. Rotation is a spelling, not a part

The corpus writes the same brick both ways round. `1x2` and `2x1` are one
inventory item; treating them as two would double the vocabulary and halve
every count — a silent error, because nothing downstream would complain.


In [ ]:
raw = eda['raw_spellings']
canonical = eda['canonical_parts']
print(f"{len(raw)} raw spellings -> {len(canonical)} inventory items")
print(sorted(raw))

pairs = [(s, 'x'.join(reversed(s.split('x')))) for s in sorted(raw)]
folded = {s: r for s, r in pairs if r in raw and s != r}
print(f"{len(folded)} of them are a rotation of another: {folded}")


## 2. The corpus is heavily skewed towards two parts

A model fitted here will reach for `1x2` and `2x6` first. That is a
property of the corpus, not of good building, and it belongs in the model
card rather than in a footnote.


In [ ]:
order = sorted(canonical, key=canonical.get, reverse=True)
total = sum(canonical.values())
for part in order:
    n = canonical[part]
    print(f'{part:>4}  {n:>9,}  {n / total:6.1%}')
print(f'{"total":>4}  {total:>9,}')

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(order, [canonical[p] / 1e6 for p in order], color='#C8461E')
ax.set_ylabel('millions of placements')
ax.set_title('Placements by inventory item', loc='left')
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
plt.show()


## 3. Why the counterfactual set had to be generated

Most objects have more than one recorded build. If those variants used
*different parts*, they would already be the counterfactual pairs the
inventory work needs — same target, different stock. They mostly do not.


In [ ]:
multi = eda['multi_structure_objects']
rows = [('identical inventory', eda['variants_identical_inventory']),
        ('differ in counts only', eda['variants_differ_counts_only']),
        ('differ in part types', eda['variants_differ_types'])]
for label, n in rows:
    print(f'{label:<24} {n:>7,}  {n / multi:6.1%}')
print(f'{"objects with variants":<24} {multi:>7,}')

usable = eda['variants_differ_types'] / multi
print(f'\nonly {usable:.1%} are usable as-is; the rest are re-tiled by '
      f'src/data/retile.py')


## 4. The two checks that would have sunk the whole thing

Both are recorded as zero. They are printed rather than trusted: a number
nobody looks at is a number nobody notices changing.


In [ ]:
for key in ('parse_failures', 'leaked_objects', 'structures_with_collisions',
            'oob_h_to_x'):
    print(f'{key:<28} {eda[key]:>8,}')

assert eda['parse_failures'] == 0, 'the parser dropped rows silently'
assert eda['leaked_objects'] == 0, 'an object reached two splits'
print('\nboth clean')

print(f"\noob_h_to_y is {eda['oob_h_to_y']:,} and is not an error: it is the "
      'count of bricks whose height axis maps to y, which the parser '
      'normalises rather than rejects.')


## 5. Where the constraint work starts

`data/reports/02_retile.json` is the next step: given a fixed shape and a
*restricted* inventory, can CP-SAT lay it out at all? Overall yes 88% of
the time, but the per-part spread is the interesting part.


In [ ]:
retile = report('02_retile.json')
print(f"{retile['n_structures']} structures, {retile['time_limit']:.0f}s limit, "
      f"{retile['verify_failures']} verification failures")
print(f"overall feasible: {retile['overall_feasible']:.1%}\n")
for part, row in sorted(retile['by_part'].items(),
                        key=lambda kv: -kv[1]['feasible_rate']):
    print(f"{part:>4}  {row['feasible_rate']:6.1%}  median {row['median_s']:.3f}s")


---

Figures for the README are generated from these same files by
`scripts/73_figures.py` (`make figures`), so a chart and this notebook
cannot disagree.
